###**Preparar entorno: clonar repo e instalar dependencias**

In [2]:
# Trae el repo al runtime de Colab e instala librerías del requirements.txt.

BRANCH = "fase-0-estructura"

!git clone -b {BRANCH} --single-branch https://github.com/Juan-Draghi/relevamiento-boletin-oficial-caba-con-llm
%cd relevamiento-boletin-oficial-caba-con-llm

!pip install -r requirements.txt

# Crear carpetas por si faltan en el runtime
!mkdir -p data/raw data/processed data/labels models reports app scripts

Cloning into 'relevamiento-boletin-oficial-caba-con-llm'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 99 (delta 47), reused 42 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 3.58 MiB | 15.74 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/relevamiento-boletin-oficial-caba-con-llm/relevamiento-boletin-oficial-caba-con-llm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 76.6 M

### **Imports del pipeline y chequeo de rutas**

In [ ]:
# Importación de utilidades y ubicación de las carpetas de entrada/salida.

import pandas as pd
from pathlib import Path

from src.config import DATA_RAW, DATA_PROCESSED
from src.candidates import run_batch

print("📥 DATA_RAW      :", DATA_RAW)
print("📤 DATA_PROCESSED:", DATA_PROCESSED)
DATA_RAW, DATA_PROCESSED

###**Configuración de corrida**

In [ ]:
# Define parámetros de ejecución (filtro opcional por verbos de acción y rutas de salida)

FILTER_HAS_ACTION = False  # True = solo guarda candidatos con verbo de acción normativa
OUT_CSV    = DATA_PROCESSED / "candidatos.csv"
MASTER_CSV = DATA_PROCESSED / "dataset_master.csv"

print("Filtro has_action:", FILTER_HAS_ACTION)
print("Salida corrida   :", OUT_CSV)
print("Maestro acumulado:", MASTER_CSV)

###**Ejecutar el pipeline (PDFs → candidatos)**

In [ ]:
# Recorre todos los PDFs en data/raw/, genera candidatos.csv y acumula en dataset_master.csv.

run_batch(
    input_dir=DATA_RAW,
    out_csv=OUT_CSV,
    filter_has_action=FILTER_HAS_ACTION,
    master_csv=MASTER_CSV
)
print("✅ Ejecución completa.")

###**Vista rápida de resultados de esta corrida**

In [ ]:
# Carga y muestra una muestra de candidatos.csv (corrida actual)

df = pd.read_csv(OUT_CSV) if OUT_CSV.exists() else pd.DataFrame()
print("Filas en candidatos.csv:", len(df))
display(df.head(10) if len(df) else df)

###**Estado del maestro y chequeos básicos**

In [ ]:
# Revisa el acumulado histórico, balance por has_action y duplicados por hash de contexto

dm = pd.read_csv(MASTER_CSV) if MASTER_CSV.exists() else pd.DataFrame()
print("Filas en dataset_master.csv:", len(dm))

if len(dm):
    display(dm.sample(min(10, len(dm))))
    print("\nBalance por has_action:")
    print(dm["has_action"].value_counts(dropna=False))
    if "context_hash" in dm.columns:
        dups = dm["context_hash"].duplicated().sum()
        print("Duplicados por context_hash en maestro:", dups)
